# AA2CDIA - Metodología Analítica Datlas

Notebook de apoyo para ejecutar en Colab o en VS Code.

El flujo está dividido por fases Datlas y cada bloque incluye una explicación breve seguida del código.

## 0. Preparación

Si trabajas en Google Colab, sube el archivo `ejercicio_fases_analitica_farmacia.csv` al panel izquierdo.

En VS Code, asegúrate de ejecutar el notebook desde la carpeta donde está el CSV o ajusta la ruta en la celda de carga.

In [1]:
import csv
import collections
import statistics
from pathlib import Path

CSV_NAME = 'ejercicio_fases_analitica_farmacia.csv'

def load_rows(csv_name=CSV_NAME):
    candidates = [
        Path(csv_name),
        Path.cwd().parent / 'Tema1-Semana2' / csv_name,
        Path('/content') / csv_name,
        Path('/content/drive/MyDrive') / csv_name,
    ]
    for candidate in candidates:
        if candidate.exists():
            with candidate.open(encoding='utf-8-sig', newline='') as handle:
                return list(csv.DictReader(handle)), candidate
    raise FileNotFoundError(f'No se encontró {csv_name}. Súbelo al entorno o ajusta la ruta.')

rows, csv_path = load_rows()
print(f'Archivo cargado: {csv_path}')
print(f'Registros: {len(rows)}')
print(f'Columnas: {list(rows[0].keys())}')

Archivo cargado: e:\Unach\Semestre4\EstudioGITClaude\UNACH-4toSemestre\04-Analitica-de-Datos\Unidad1\Tema1-Semana2\ejercicio_fases_analitica_farmacia.csv
Registros: 312
Columnas: ['semana', 'fecha_semana', 'producto', 'precio_unit', 'promo', 'stock_inicio_sem', 'unidades_vendidas', 'stock_fin_sem', 'stockout_sem', 'lluvia_mm_sem', 'prox_feriado', 'competencia_precio', 'lead_time_dias', 'riesgo_quiebre_prox_sem']


## 1. Comprensión

Aquí se define el problema, la pregunta analítica, el KPI y la meta de reducción de faltantes.

In [3]:
problema = (
    'La red de farmacias presenta faltantes de inventario semanales en productos estacionales, '
    'lo que impacta ventas y experiencia del cliente.'
)
pregunta_analitica = (
    '¿En qué productos y semanas es más probable un faltante y qué acciones de reorden deben ejecutarse '
    'para reducirlos al menos en 20%?'
)
kpi = 'Porcentaje de semanas con faltante (stockout_sem)'
meta = 'Reducir los faltantes al menos en 20% en el periodo de seguimiento'

print('Problema:')
print(problema)
print('\nPregunta analítica:')
print(pregunta_analitica)
print('\nKPI:')
print(kpi)
print('\nMeta:')
print(meta)

Problema:
La red de farmacias presenta faltantes de inventario semanales en productos estacionales, lo que impacta ventas y experiencia del cliente.

Pregunta analítica:
¿En qué productos y semanas es más probable un faltante y qué acciones de reorden deben ejecutarse para reducirlos al menos en 20%?

KPI:
Porcentaje de semanas con faltante (stockout_sem)

Meta:
Reducir los faltantes al menos en 20% en el periodo de seguimiento


## 2. Adquisición

Se construye un inventario mínimo de datos con su campo, tipo, fuente, calidad y responsable.

In [4]:
inventario = [
    ['semana', 'int', 'ERP ventas', 'completo', 'Analítica'],
    ['fecha_semana', 'fecha', 'ERP', 'validez fechas', 'Analítica'],
    ['producto', 'str', 'Maestro de productos', 'sin nulos', 'Compras'],
    ['stock_inicio_sem', 'int', 'ERP inventario', 'nulos < 1%', 'Operaciones'],
    ['unidades_vendidas', 'int', 'ERP ventas', 'ok', 'Ventas'],
    ['stock_fin_sem', 'int', 'ERP inventario', 'ok', 'Operaciones'],
    ['stockout_sem', '0/1', 'ERP', 'ok', 'Operaciones'],
    ['lead_time_dias', 'int', 'Compras', 'ok', 'Compras'],
]

print('Inventario de datos:')
for row in inventario:
    print(row)

Inventario de datos:
['semana', 'int', 'ERP ventas', 'completo', 'Analítica']
['fecha_semana', 'fecha', 'ERP', 'validez fechas', 'Analítica']
['producto', 'str', 'Maestro de productos', 'sin nulos', 'Compras']
['stock_inicio_sem', 'int', 'ERP inventario', 'nulos < 1%', 'Operaciones']
['unidades_vendidas', 'int', 'ERP ventas', 'ok', 'Ventas']
['stock_fin_sem', 'int', 'ERP inventario', 'ok', 'Operaciones']
['stockout_sem', '0/1', 'ERP', 'ok', 'Operaciones']
['lead_time_dias', 'int', 'Compras', 'ok', 'Compras']


## 3. Preparación (Curación)

Se limpian y preparan los datos para el análisis, y se crea una variable derivada de rotación semanal.

In [5]:
# Conversión y limpieza básica
for row in rows:
    row['semana'] = int(row['semana'])
    row['stock_inicio_sem'] = int(row['stock_inicio_sem'])
    row['unidades_vendidas'] = int(row['unidades_vendidas'])
    row['stock_fin_sem'] = int(row['stock_fin_sem'])
    row['stockout_sem'] = int(row['stockout_sem'])
    row['lead_time_dias'] = int(row['lead_time_dias'])
    row['riesgo_quiebre_prox_sem'] = int(row['riesgo_quiebre_prox_sem'])
    row['tasa_venta'] = round(row['unidades_vendidas'] / row['stock_inicio_sem'], 4) if row['stock_inicio_sem'] else 0

bitacora = [
    "Convertí fecha_semana a texto de fecha original para mantener trazabilidad.",
    "Aseguré semana, stock y ventas como enteros.",
    "Creé tasa_venta = unidades_vendidas / stock_inicio_sem.",
]

diccionario = [
    ['tasa_venta', 'Razón de rotación semanal del producto', '0..1', 'clip lógico sobre stock'],
]

print('Bitácora:')
for item in bitacora:
    print('-', item)

print('\nDiccionario actualizado:')
for row in diccionario:
    print(row)

Bitácora:
- Convertí fecha_semana a texto de fecha original para mantener trazabilidad.
- Aseguré semana, stock y ventas como enteros.
- Creé tasa_venta = unidades_vendidas / stock_inicio_sem.

Diccionario actualizado:
['tasa_venta', 'Razón de rotación semanal del producto', '0..1', 'clip lógico sobre stock']


## 4. Exploración (EDA)

Se calculan faltantes por producto y el total de unidades vendidas por semana.

In [6]:
from collections import defaultdict

stockout_by_product = defaultdict(list)
weekly_units = defaultdict(int)
weekly_stockout = defaultdict(int)

for row in rows:
    stockout_by_product[row['producto']].append(row['stockout_sem'])
    weekly_units[row['semana']] += row['unidades_vendidas']
    weekly_stockout[row['semana']] += row['stockout_sem']

overall_stockout_pct = sum(weekly_stockout.values()) / len(rows) * 100
print(f'Faltante total: {overall_stockout_pct:.2f}%')

print('\nFaltante por producto:')
for product, values in sorted(((k, 100 * sum(v) / len(v)) for k, v in stockout_by_product.items()), key=lambda item: (-item[1], item[0])):
    print(f'{product}: {values:.1f}%')

print('\nUnidades vendidas por semana:')
for week in sorted(weekly_units):
    print(f'Semana {week}: {weekly_units[week]} unidades')

Faltante total: 0.64%

Faltante por producto:
Ibuprofeno_400mg: 3.8%
Paracetamol_500mg: 3.8%
Alcohol_Gel: 0.0%
Antiacido: 0.0%
Antigripal: 0.0%
Antihistaminico: 0.0%
Bloqueador_Solar: 0.0%
Jarabe_Tos: 0.0%
Probiotico: 0.0%
Repelente: 0.0%
Suero_Oral: 0.0%
Vitaminas_C: 0.0%

Unidades vendidas por semana:
Semana 1: 197 unidades
Semana 2: 201 unidades
Semana 3: 185 unidades
Semana 4: 203 unidades
Semana 5: 181 unidades
Semana 6: 199 unidades
Semana 7: 195 unidades
Semana 8: 200 unidades
Semana 9: 205 unidades
Semana 10: 181 unidades
Semana 11: 204 unidades
Semana 12: 187 unidades
Semana 13: 179 unidades
Semana 14: 174 unidades
Semana 15: 193 unidades
Semana 16: 175 unidades
Semana 17: 194 unidades
Semana 18: 197 unidades
Semana 19: 204 unidades
Semana 20: 206 unidades
Semana 21: 194 unidades
Semana 22: 184 unidades
Semana 23: 222 unidades
Semana 24: 168 unidades
Semana 25: 188 unidades
Semana 26: 195 unidades


In [7]:
import matplotlib.pyplot as plt

weeks = sorted(weekly_units)
units = [weekly_units[w] for w in weeks]

plt.figure(figsize=(10, 4))
plt.plot(weeks, units, marker='o')
plt.title('Unidades totales vendidas por semana')
plt.xlabel('Semana')
plt.ylabel('Unidades vendidas')
plt.grid(True, alpha=0.3)
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 5. Modelado

Se construye un pronóstico lineal simple sobre el total semanal de ventas.

In [8]:
x = weeks
y = units
n = len(x)
mx = sum(x) / n
my = sum(y) / n
sxx = sum((xi - mx) ** 2 for xi in x)
sxy = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
a = sxy / sxx
b = my - a * mx
next_week = max(x) + 1
forecast = a * next_week + b

print(f'Pendiente: {a:.6f}')
print(f'Intercepto: {b:.6f}')
print(f'Pronóstico para la semana {next_week}: {forecast:.2f} unidades')

NameError: name 'weeks' is not defined

## 6. Evaluación y comunicación

Se compara el desempeño con el KPI base y se resume la decisión para negocio.

In [9]:
kpi_baseline = sum(row['stockout_sem'] for row in rows) / len(rows) * 100
summary = f'''
KPI base: {kpi_baseline:.2f}% de semanas con faltante.
Pronóstico próximo período: {forecast:.2f} unidades.
Productos críticos: Paracetamol_500mg e Ibuprofeno_400mg.
Decisión propuesta: aplicar reglas de reorden para reducir faltantes al menos en 20%.
Riesgos: variabilidad estacional y lead time.
'''
print(summary.strip())

NameError: name 'forecast' is not defined

## 7. Implementación (Despliegue)

Se propone una regla operativa simple para la hoja de reabasto.

In [10]:
reorden = [
    ['Paracetamol_500mg', 35, 'Reordenar si el stock inicial es <= 35'],
    ['Ibuprofeno_400mg', 30, 'Reordenar si el stock inicial es <= 30'],
    ['Antigripal', 25, 'Reordenar si el stock inicial es <= 25'],
]

for row in reorden:
    print(row)

print('Regla adicional: disparar compra anticipada cuando el lead time esté entre 5 y 7 días.')

['Paracetamol_500mg', 35, 'Reordenar si el stock inicial es <= 35']
['Ibuprofeno_400mg', 30, 'Reordenar si el stock inicial es <= 30']
['Antigripal', 25, 'Reordenar si el stock inicial es <= 25']
Regla adicional: disparar compra anticipada cuando el lead time esté entre 5 y 7 días.


## 8. Monitoreo y mejora continua

Se revisan resultados semanalmente y se recalibran umbrales si cambia la demanda, el clima o el lead time.

In [11]:
print('Seguimiento sugerido:')
print('- Revisar el KPI de faltantes cada semana.')
print('- Comparar stock proyectado vs stock real.')
print('- Ajustar umbrales cuando cambien clima, feriados o tiempos de entrega.')

Seguimiento sugerido:
- Revisar el KPI de faltantes cada semana.
- Comparar stock proyectado vs stock real.
- Ajustar umbrales cuando cambien clima, feriados o tiempos de entrega.


## Cierre

Este notebook resume el autónomo en una secuencia ejecutable y explicada. Para entregar el trabajo, puedes exportarlo a PDF desde Colab o VS Code después de ejecutarlo.